In [2]:
import pandas as pd
import inspect

from business_entity_resolution.components.data_ingestion import DataIngestion
from business_entity_resolution.components.data_transformation import DataTransformation
from business_entity_resolution.components.candidate_generation import CandidateGeneration


In [3]:
import os

PROJECT_ROOT = r"C:\Users\kaila\OneDrive\Desktop\Amazon ML"

os.chdir(PROJECT_ROOT)

print("Working directory:")
print(os.getcwd())

Working directory:
C:\Users\kaila\OneDrive\Desktop\Amazon ML


In [4]:
ingestion = DataIngestion()

source1, source2, source3, ground_truth = ingestion.initiate_data_ingestion()

print("Source 1:", source1.shape)
print("Source 2:", source2.shape)
print("Source 3:", source3.shape)
print("Ground truth:", ground_truth.shape)


Source 1: (2206821, 4)
Source 2: (5034616, 4)
Source 3: (5285603, 4)
Ground truth: (2206821, 2)


In [5]:
print(inspect.getsource(CandidateGeneration.create_blocking_keys))


    @staticmethod
    def create_blocking_keys(df):

        df = df.copy()

        # -----------------------------
        # Name-based blocking
        # -----------------------------

        df["name_block"] = (
            df["business_name_clean"]
            .fillna("")
            .astype(str)
            .str[:4]
        )

        df["name_blocking_key"] = (
            df["country"]
            .fillna("")
            .astype(str)
            .str.lower()
            + "_"
            + df["name_block"]
        )

        # -----------------------------
        # Address-based blocking
        # -----------------------------

        df["address_block"] = (
            df["business_address_clean"]
            .fillna("")
            .astype(str)
            .str[:6]
        )

        df["address_blocking_key"] = (
            df["country"]
            .fillna("")
            .astype(str)
            .str.lower()
            + "_"
            + df["address_block"]
        )

In [6]:
matched_s1_ids = (
    ground_truth[
        ground_truth["matched_entity_ids"].notna()
        & ground_truth["matched_entity_ids"].ne("")
    ]["source1_entity_id"]
    .drop_duplicates()
)

selected_s1_ids = (
    matched_s1_ids
    .sample(n=100, random_state=42)
    .tolist()
)

small_source1 = source1[
    source1["entity_id"].isin(selected_s1_ids)
].copy()

small_ground_truth = ground_truth[
    ground_truth["source1_entity_id"].isin(selected_s1_ids)
].copy()

print("Selected S1:", len(selected_s1_ids))


Selected S1: 100


In [7]:
def get_matching_ids(ground_truth_df, source_prefix):
    matched_ids = (
        ground_truth_df["matched_entity_ids"]
        .dropna()
        .astype(str)
        .str.split(",")
        .explode()
        .str.strip()
    )

    return matched_ids[
        matched_ids.str.startswith(source_prefix)
    ].unique()


true_s2_ids = get_matching_ids(
    small_ground_truth,
    "S2-"
)

true_s3_ids = get_matching_ids(
    small_ground_truth,
    "S3-"
)

small_source2_true = source2[
    source2["entity_id"].isin(true_s2_ids)
].copy()

small_source3_true = source3[
    source3["entity_id"].isin(true_s3_ids)
].copy()

print("True S2 records:", len(true_s2_ids))
print("True S3 records:", len(true_s3_ids))


True S2 records: 195
True S3 records: 179


In [8]:
#add non matching records
extra_s2 = source2[
    ~source2["entity_id"].isin(true_s2_ids)
].head(max(5000 - len(small_source2_true), 0))

extra_s3 = source3[
    ~source3["entity_id"].isin(true_s3_ids)
].head(max(5000 - len(small_source3_true), 0))

small_source2 = pd.concat(
    [small_source2_true, extra_s2],
    ignore_index=True
)

small_source3 = pd.concat(
    [small_source3_true, extra_s3],
    ignore_index=True
)

print("S1 sample:", small_source1.shape)
print("S2 sample:", small_source2.shape)
print("S3 sample:", small_source3.shape)


S1 sample: (100, 4)
S2 sample: (5000, 4)
S3 sample: (5000, 4)


In [9]:
transformer = DataTransformation()

small_source1 = transformer.transform_source(small_source1)
small_source2 = transformer.transform_source(small_source2)
small_source3 = transformer.transform_source(small_source3)

print("Transformation complete.")


Transformation complete.


In [10]:
#genarte candidates
candidate_generator = CandidateGeneration()

candidates_s2, candidates_s3 = (
    candidate_generator.generate_candidates(
        small_source1,
        small_source2,
        small_source3
    )
)

print("S2 candidate pairs:", len(candidates_s2))
print("S3 candidate pairs:", len(candidates_s3))


S2 candidate pairs: 1021
S3 candidate pairs: 1174


In [11]:
#get true pairs
ground_truth_pairs = (
    small_ground_truth[
        ["source1_entity_id", "matched_entity_ids"]
    ]
    .dropna(subset=["matched_entity_ids"])
    .copy()
)

ground_truth_pairs["matched_entity_id"] = (
    ground_truth_pairs["matched_entity_ids"]
    .str.split(",")
)

ground_truth_pairs = (
    ground_truth_pairs
    .explode("matched_entity_id")
)

ground_truth_pairs["matched_entity_id"] = (
    ground_truth_pairs["matched_entity_id"]
    .str.strip()
)

ground_truth_pairs = ground_truth_pairs[
    ground_truth_pairs["matched_entity_id"].ne("")
].drop_duplicates()

true_s2_pairs = ground_truth_pairs[
    ground_truth_pairs["matched_entity_id"].str.startswith("S2-")
].copy()

true_s3_pairs = ground_truth_pairs[
    ground_truth_pairs["matched_entity_id"].str.startswith("S3-")
].copy()

print("Total true pairs:", len(ground_truth_pairs))
print("True S2 pairs:", len(true_s2_pairs))
print("True S3 pairs:", len(true_s3_pairs))


Total true pairs: 374
True S2 pairs: 195
True S3 pairs: 179


In [12]:
def calculate_blocking_recall(
    true_pairs,
    candidate_pairs,
    candidate_id_column
):
    true_set = set(
        zip(
            true_pairs["source1_entity_id"],
            true_pairs["matched_entity_id"]
        )
    )

    candidate_set = set(
        zip(
            candidate_pairs["entity_id_s1"],
            candidate_pairs[candidate_id_column]
        )
    )

    found_pairs = true_set.intersection(candidate_set)

    return {
        "true_pairs": len(true_set),
        "candidate_pairs": len(candidate_set),
        "true_pairs_found": len(found_pairs),
        "true_pairs_missed": len(true_set - candidate_set),
        "blocking_recall": (
            len(found_pairs) / len(true_set)
            if true_set else 0.0
        )
    }


s2_result = calculate_blocking_recall(
    true_s2_pairs,
    candidates_s2,
    "entity_id_s2"
)

s3_result = calculate_blocking_recall(
    true_s3_pairs,
    candidates_s3,
    "entity_id_s3"
)

print("====================================")
print("BLOCKING RECALL RESULTS")
print("====================================")
print(f"S2 Blocking Recall: {s2_result['blocking_recall']:.4f}")
print(f"S3 Blocking Recall: {s3_result['blocking_recall']:.4f}")
print(
    f"S2 True Pairs Found: "
    f"{s2_result['true_pairs_found']} / {s2_result['true_pairs']}"
)
print(
    f"S3 True Pairs Found: "
    f"{s3_result['true_pairs_found']} / {s3_result['true_pairs']}"
)


BLOCKING RECALL RESULTS
S2 Blocking Recall: 0.9436
S3 Blocking Recall: 0.9441
S2 True Pairs Found: 184 / 195
S3 True Pairs Found: 169 / 179


In [13]:
s2_true_set = set(
    zip(
        true_s2_pairs["source1_entity_id"],
        true_s2_pairs["matched_entity_id"]
    )
)

s2_candidate_set = set(
    zip(
        candidates_s2["entity_id_s1"],
        candidates_s2["entity_id_s2"]
    )
)

missed_s2_pairs = s2_true_set - s2_candidate_set

missed_s2_df = pd.DataFrame(
    list(missed_s2_pairs),
    columns=[
        "source1_entity_id",
        "matched_entity_id"
    ]
)

print("Missed S2 pairs:", len(missed_s2_df))
missed_s2_df.head(20)


Missed S2 pairs: 11


,source1_entity_id,matched_entity_id
0,S1-782734433,S2-317096646
1,S1-782734433,S2-609366455
2,S1-782734433,S2-508961118
3,S1-826085565,S2-462148816
4,S1-782734433,S2-542083228
5,S1-354889538,S2-334019354
6,S1-826085565,S2-69491551
7,S1-134755367,S2-214030604
8,S1-491000076,S2-193933459
9,S1-782734433,S2-31561982


In [14]:
missed_s2_details = (
    missed_s2_df
    .merge(
        small_source1[
            [
                "entity_id",
                "business_name",
                "business_address",
                "country"
            ]
        ],
        left_on="source1_entity_id",
        right_on="entity_id",
        how="left"
    )
    .drop(columns=["entity_id"])
    .merge(
        small_source2[
            [
                "entity_id",
                "business_name",
                "business_address",
                "country"
            ]
        ],
        left_on="matched_entity_id",
        right_on="entity_id",
        how="left",
        suffixes=("_s1", "_s2")
    )
    .drop(columns=["entity_id"])
)

missed_s2_details.head(20)


,source1_entity_id,matched_entity_id,business_name_s1,business_address_s1,country_s1,business_name_s2,business_address_s2,country_s2
0,S1-782734433,S2-317096646,Tirupati Solutions LLP,"Thane, 1103 A-Tower, Maharashtra, 1St Podium, ...",India,तिरुपति सॉल्यूशंस एलएलपी,"NO 01103 A-TOWER, THANE, BHAYANDER EAST, Mahar...",India
1,S1-782734433,S2-609366455,Tirupati Solutions LLP,"Thane, 1103 A-Tower, Maharashtra, 1St Podium, ...",India,तिरुपति सॉल्यूशंस एलएलपी,"NO 01103 A-TOWER, 1ST PODIUM, SAMRIDDHI (S), I...",India
2,S1-782734433,S2-508961118,Tirupati Solutions LLP,"Thane, 1103 A-Tower, Maharashtra, 1St Podium, ...",India,तिरुपति सॉल्यूशंस एलएलपी,"#01103 A-TOWER, 1ST PODIUM, SAMRIDDHI (S), IND...",India
3,S1-826085565,S2-462148816,Om Infotech Private Limited,"Shop No. 103, First Floor, Universal Plaza 13/...",India,ominfotech.com,"FIRST FLOOR, UNIVERSAL PLAZA 13/20, W.E.A, Del...",India
4,S1-782734433,S2-542083228,Tirupati Solutions LLP,"Thane, 1103 A-Tower, Maharashtra, 1St Podium, ...",India,तिरुपति सॉल्यूशंस एलएलपी,"1ST PODIUM, SAMRIDDHI (S), INDRALOK, Maharasht...",India
5,S1-354889538,S2-334019354,Rabun Boral LLC,"226 87 Street, New York, NY",US,Xyloquoevo,"0226 87 ST, NEW YORK, NY",US
6,S1-826085565,S2-69491551,Om Infotech Private Limited,"Shop No. 103, First Floor, Universal Plaza 13/...",India,ओम इंफोटेक प्राइवेट लिमिटेड,"FIRST FLOOR, UNIVERSAL PLAZA 13/20, W.E.A, #01...",India
7,S1-134755367,S2-214030604,Om Consultancy Pvt Ltd,"23 G Floor, World Trade Centre, Barakhamba Ave...",India,ओम कंसल्टेंसी प्रा. लि.,"G FLOOR, NEW DELHI, Delhi",India
8,S1-491000076,S2-193933459,Minerva Trading Private Limited,"Plot No.49, Commercial Mandi Near Cat Square, ...",India,Mr Minerva Private Limited Service,"H.NO #49 , COMMERCIAL MANDI NEAR CAT SQUARE, R...",India
9,S1-782734433,S2-31561982,Tirupati Solutions LLP,"Thane, 1103 A-Tower, Maharashtra, 1St Podium, ...",India,तिरुपति सॉल्यूशंस एलएलपी,"महाराष्ट्र, BHAYANDER EAST, 1ST PODIUM, SAMRID...",India
